In [ ]:
# This note book follows a tutorial from https://www.datacamp.com/tutorial/xgboost-in-python

In [11]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

In [ ]:
warnings.filterwarnings("ignore")

diamonds = sns.load_dataset("diamonds")

diamonds.head()

In [ ]:
diamonds.info()
diamonds.shape

In [ ]:
diamonds.describe()

In [ ]:
diamonds.describe(exclude=np.number)

In [24]:
# In this example we will first try to predict diamond prices using their physical measurements, so out target will be the price column.
# The function below is used to split the data into training and testing sets for ML models

from sklearn.model_selection import train_test_split

# extract feature and target arrays from the data frame called diamonds
# The drop method is used to remove the price column from the data fram 
# and the result is assigned to the variable X
# The price column is assigned to the variable y using the syntax diamonds["price"]
X, y = diamonds.drop("price", axis=1), diamonds["price"]

In [17]:
# Extract text features
cats = X.select_dtypes(exclude=np.number).columns.tolist()

# Convert to Pandas category
for col in cats:
   X[col] = X[col].astype('category')

In [ ]:
X.dtypes

In [19]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

In [21]:
# The code below imports the xgboost library and creates regression matrices from the training and testing sets using the xgb.DMatrix() function
# dtrain = regression matrix for the training set
# dtest = regression matrix for the testing set
# The enable_categorical=True parameter is used to enable categorical features in the regression matrices
# The xgb.DMatrix() function is used to convert the input data into an internal data structure that can be used by the XGBoost algorithm
# This function takes the input data as its first argument, and the target variable as its second argument. 
# input_data = X_train, target_variable = y_train
# input_data = X_test, target_variable = y_test
# Overall, this code prepares the data for use in an XGBoost regression model by creating matrices that can be used as input to the algorithm

import xgboost as xgb

# Create regression matrices
dtrain_reg = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_reg = xgb.DMatrix(X_test, y_test, enable_categorical=True)

In [ ]:
# After building the DMatrices, you should choose a value for the objective parameter
# It tells XGBoost the machine learning problem you are trying to solve and what metrics or loss functions to use to solve that problem
# For example, to predict diamond prices, which is a regression problem, you can use the common reg:squarederror objective
#  Usually, the name of the objective also contains the name of the loss function for the problem 
# For regression, it is common to use Root Mean Squared Error, which minimizes the square root of the squared sum of the differences between actual and predicted values
# Here is how the metric would look like when implemented in NumPy:
# mse = np.mean((actual - predicted) ** 2)
# rmse = np.sqrt(mse)

In [ ]:
# Define hyperparameters notes
# params = {"objective": "reg:squarederror", "tree_method": "gpu_hist"}
# The first hyperparameter is "objective": "reg:squarederror", which specifies that the model will use mean squared error as the loss function for regression
# The second hyperparameter is "tree_method": "gpu_hist", which specifies that the model will use the GPU to build the trees
# This can significantly speed up the training process for large datasets.
# Inside this initial params, we are also setting tree_method to gpu_hist, which enables GPU acceleration
# If you don't have a GPU, you can omit the parameter or set it to hist

# Now, we set another parameter called num_boost_round, which stands for number of boosting rounds
# Internally, XGBoost minimizes the loss function RMSE in small incremental rounds (more on this later)
# This parameter specifies the amount of those rounds
# The ideal number of rounds is found through hyperparameter tuning. For now, we will just set it to 100:

In [47]:
# Define hyperparameters
# model using mres as the loss function (What other loss functions can be used?= mae, huber, logistic, poisson, quantile, etc.)
# model using hist as the tree method (What other tree methods can be used? = gpu_hist, auto?, exact?, approx?)
params = {"objective": "reg:squarederror", "tree_method": "hist"}

n = 100
model = xgb.train(
   params=params,
   dtrain=dtrain_reg,
   num_boost_round=n,
)

In [48]:
# Evaluation

# During the boosting rounds, the model object has learned all the patterns of the training set it possibly can 
# Now, we must measure its performance by testing it on unseen data
# That's where our dtest_reg DMatrix comes into play:
# So here the trained model is used to predict the prices of the diamonds in the testing set 

from sklearn.metrics import mean_squared_error, mean_absolute_error

preds = model.predict(dtest_reg)


In [49]:
# This step of the process is called model evaluation (or inference)
# Once you generate predictions with predict, you pass them inside mean_squared_error function of Sklearn to compare against y_test:
# The code below calculate the RMSE of the model
# module is used to calculate the mean squared error between the true values (y_test) and the predicted values (preds)
#  Good result is a low RMSE value
# I also used the mean_absolute_error function to calculate the mean absolute error between the true values and the predicted values
# The mean absolute error is another metric used to evaluate regression models
# It is the average of the absolute differences between the predicted and true values
# The lower the mean absolute error, the better the model

rmse = mean_squared_error(y_test, preds, squared=False)
# mae = mean_absolute_error(y_test, preds)

print(f"RMSE of the base model: {rmse:.3f}")
# print(f"MAE of the base model: {mae:.3f}")

RMSE of the base model: 552.861


In [50]:
# Next, we create a list of two tuples that each contain two elements
# The first element is the array for the model to evaluate, and the second is the array’s name
# The first tuple contains the training set, and the second tuple contains the testing set
# When we pass this array to the evals parameter of xgb.train, we will see the model performance after each boosting round:

evals = [(dtrain_reg, "train"), (dtest_reg, "validation")]

model = xgb.train(
   params=params,
   dtrain=dtrain_reg,
   num_boost_round=n,
   evals=evals,
   verbose_eval= 10 # This parameter controls how often the model prints the evaluation metrics in this case every 10 rounds
)

[0]	train-rmse:2874.49146	validation-rmse:2817.90814
[10]	train-rmse:548.36512	validation-rmse:592.03160
[20]	train-rmse:491.09887	validation-rmse:558.53485
[30]	train-rmse:469.58201	validation-rmse:555.51015
[40]	train-rmse:454.32953	validation-rmse:554.45666
[50]	train-rmse:438.68033	validation-rmse:554.13365
[60]	train-rmse:425.38361	validation-rmse:551.57888
[70]	train-rmse:414.71115	validation-rmse:549.26109
[80]	train-rmse:405.41008	validation-rmse:549.03952
[90]	train-rmse:391.04269	validation-rmse:551.87206
[99]	train-rmse:383.48826	validation-rmse:552.86131
